<a href="https://colab.research.google.com/github/divyankapawaskar/splink-workshop-binder/blob/main/Splink_synthetic_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [68]:
!pip install splink

# Import necessary python packages
import pandas as pd
import splink
import numpy as np


In [4]:

#### Import datasets to be linked ----------------------------------------------

# Load NY crash dataset
#df_crash = pd.read_csv("/synthetic_crash.csv", index_col = 0)
df_crash = pd.read_csv("/content/synthetic_crash.csv", index_col = 0)

# Load NY hospital records dataset
#df_hosp = pd.read_csv("Synthetic Dataset/synthetic hosp dataset.csv", index_col = 0)
df_hosp = pd.read_csv("/content/synthetic_hosp.csv", index_col = 0)

# View crash and hcup data
df_crash.head
df_hosp.head

# Check dimensions of datasets
print(f"\nCrash records: {df_crash.shape[0]} rows x {df_crash.shape[1]} columns")
print(f"Hospital records: {df_hosp.shape[0]} rows x {df_hosp.shape[1]} columns")



Crash records: 1000 rows x 11 columns
Hospital records: 1000 rows x 11 columns


In [5]:
# Step 3: Rename columns so both datasets share the same column names for linkage
# Crash columns: record_id, crash_id, first_name, last_name, sex, age,
#                date_of_birth, role, crash_date, crash_hour, county, state,
#                vehicle_type, vehicle_make

df_crash.rename(columns={
    'record_id':    'unique_id',
    'first_name':   'firstnamef2',
    'last_name':    'lastnamef2',
    'gender':          'sex',
    'age':          'age',
    'bday':'dob',
    'crash_date':   'admit_date',
    'crash_hour':   'admit_hr',
    'county':       'county',
    'role':         'role',
    'state':        'state',
}, inplace=True)

# Hospital columns: patient_record_id, first_name, last_name, sex, age,
#                   date_of_birth, arrival_date, arrival_hour, county, hospital_name

df_hosp.rename(columns={
    'patient_record_id': 'unique_id',
    'first_name':        'firstnamef2',
    'last_name':         'lastnamef2',
    'gender':               'sex',
    'age':               'age',
    'bday':     'dob',
    'crash_date':      'admit_date',
    'crash_hour':      'admit_hr',
    'county':            'county',
}, inplace=True)

# Check that column names match
df_crash.head(2)
df_hosp.head(2)


,firstnamef2,lastnamef2,sex,age,dob,role,admit_date,admit_hr,county,state,zip
record_id,,,,,,,,,,,
1,Kenneth,Davis,Male,19.0,03/24/2005,Passenger,2024-02-24,0,WESTCHESTER,NY,NaN
2,Edward,Thompson,Female,55.0,09/04/1969,Driver,2024-07-14,5,QUEENS,NY,14604.0


In [6]:
# Step 4: Check and convert data types for each column
# Best practice is to convert everything to strings or numeric types
print(df_crash.dtypes)
print(df_hosp.dtypes)



# Step 5: Convert columns to correct types
string_cols = ['firstnamef2', 'lastnamef2', 'sex', 'dob', 'admit_date', 'county', 'role', 'state']
int_cols    = ['age', 'admit_hr', 'zip']

for col in string_cols:
    if col in df_crash.columns:
        df_crash[col] = df_crash[col].astype('string')
    if col in df_hosp.columns:
        df_hosp[col] = df_hosp[col].astype('string')

for col in int_cols:
    if col in df_crash.columns:
        df_crash[col] = pd.to_numeric(df_crash[col], errors='coerce').astype('Int64')
    if col in df_hosp.columns:
        df_hosp[col] = pd.to_numeric(df_hosp[col], errors='coerce').astype('Int64')

print(df_crash.dtypes)
print(df_hosp.dtypes)

#### Ensure datasets have matching fields, column names, and data types---------

firstnamef2     object
lastnamef2      object
sex             object
age            float64
dob             object
role            object
admit_date      object
admit_hr         int64
county          object
state           object
zip            float64
dtype: object
firstnamef2     object
lastnamef2      object
sex             object
age            float64
dob             object
role            object
admit_date      object
admit_hr         int64
county          object
state           object
zip            float64
dtype: object
firstnamef2    string[python]
lastnamef2     string[python]
sex            string[python]
age                     Int64
dob            string[python]
role           string[python]
admit_date     string[python]
admit_hr                Int64
county         string[python]
state          string[python]
zip                     Int64
dtype: object
firstnamef2    string[python]
lastnamef2     string[python]
sex            string[python]
age                     Int64
do

In [8]:
# Step 6
#Match crash and hosp sex

df_hosp['sex'] = df_hosp['sex'].astype('string').replace({'Male': 'M', 'Female': 'F'})

#lowercase 'county'

for _df in (df_crash, df_hosp):
    _df['county'] = _df['county'].str.lower()



#Standardize date formats
df_crash['admit_date'] = pd.to_datetime(df_crash['admit_date'], errors='coerce').dt.strftime('%Y-%m-%d')
df_hosp['admit_date']  = pd.to_datetime(df_hosp['admit_date'],  errors='coerce').dt.strftime('%Y-%m-%d')

df_crash['dob'] = pd.to_datetime(df_crash['dob'], errors='coerce').dt.strftime('%Y-%m-%d')
df_hosp['dob']  = pd.to_datetime(df_hosp['dob'],  errors='coerce').dt.strftime('%Y-%m-%d')

# %%
# Step 7: Add source markers and splink IDs
df_crash['unique_id'] = 'crash_' + df_crash.index.astype(str)
df_hosp['unique_id']  = 'hosp_'  + df_hosp.index.astype(str)

df_crash['source'] = 'crash'
df_hosp['source']  = 'hosp'

# Verify that all columns are now numeric or string, and match between datasets
df_crash.dtypes
df_hosp.dtypes


,0
firstnamef2,string[python]
lastnamef2,string[python]
sex,string[python]
age,Int64
dob,object
role,string[python]
admit_date,object
admit_hr,Int64
county,string[python]
state,string[python]


In [9]:
# Import splink modules
from splink import DuckDBAPI
from splink.exploratory import completeness_chart
from splink.exploratory import profile_columns

#### Generate and inspect charts to understand your data -----------------------

# Completeness Chart -- check data missingness and completeness
completeness_chart(
    [df_crash, df_hosp],
    db_api = DuckDBAPI(),
    table_names_for_chart = ["crash", "hosp"]).save("completeness_chart.html")

# Profile Columns Charts -- shows distributions of values for each variable
profile_columns(df_crash, db_api = DuckDBAPI()).save("profile_columns_crash.html")
profile_columns(df_hosp, db_api = DuckDBAPI()).save("profile_columns_hcup.html")

# Import Splink modules
from splink import block_on
from splink.blocking_analysis import cumulative_comparisons_to_be_scored_from_blocking_rules_chart
import splink.comparison_library as cl
from splink import DuckDBAPI, Linker, SettingsCreator, block_on

# Choose API
db_api = DuckDBAPI()

In [10]:
#### Define Blocking Rules to be used in Linkage -------------------------------

# Ex: if blocking on month, we are ONLY considering matches where the "month" field matches exactly
# Good blocking rules save on computation, poor blocking rules can result in too many comparisons, potentially crashing your software

blocking_rules = [
    block_on('lastnamef2', 'age'),
    block_on('dob', 'age', 'sex'),
    block_on('dob', 'county'),
    block_on('firstnamef2', 'county'),
    block_on('firstnamef2', 'lastnamef2')
]


# Cumulative comparisons chart -- use to assess your blocking rules
# Try to keep down the cumulative number of comparisons for computational efficiency
cumulative_comparisons_to_be_scored_from_blocking_rules_chart(
    table_or_tables = [df_crash, df_hosp],
    blocking_rules = blocking_rules,
    db_api = db_api,
    link_type = 'link_only',
    unique_id_column_name = 'unique_id',
    source_dataset_column_name = 'source').save("cumulative_comparisons.html")

In [54]:
# Step 9: Define custom comparisons

from splink.comparison_library import CustomComparison

admit_date_comp = CustomComparison(
    output_column_name='admit_date_comp',
    comparison_levels=[
        {
            'sql_condition': 'admit_date_r IS NULL OR admit_date_l IS NULL',
            'label_for_charts': 'null',
            'is_null_level': True
        },
        {
            'sql_condition': (
                'admit_date_l IS NOT NULL AND admit_date_r IS NOT NULL AND '
                'CAST(admit_date_l AS DATE) = CAST(admit_date_r AS DATE)'
            ),
            'label_for_charts': 'Exact match on admit_date',
        },
        {
            'sql_condition': (
                "ABS(DATE_DIFF('day', CAST(admit_date_r AS DATE), CAST(admit_date_l AS DATE))) <= 2"
            ),
            'label_for_charts': 'admit_date within 2 days',
        },
        {
            'sql_condition': 'TRUE',
            'label_for_charts': 'all other comparisons'
        }
    ]
)

# Elapsed time between crash and admit: uses date + hour (same 48h rule as before; first match wins).
admit_hr_comp = CustomComparison(
    output_column_name='admit_hr_comp',
    comparison_levels=[
        {
            'sql_condition': (
                'admit_date_r IS NULL OR admit_date_l IS NULL OR '
                'admit_hr_r IS NULL OR admit_hr_l IS NULL'
            ),
            'label_for_charts': 'null',
            'is_null_level': True
        },
        {
            'sql_condition': (
                'admit_date_l IS NOT NULL AND admit_date_r IS NOT NULL AND '
                'admit_hr_l IS NOT NULL AND admit_hr_r IS NOT NULL AND '
                'CAST(admit_date_l AS DATE) = CAST(admit_date_r AS DATE) AND '
                'CAST(admit_hr_l AS DOUBLE) = CAST(admit_hr_r AS DOUBLE)'
            ),
            # Same idea as ExactMatch, but needs BOTH date and hour (hour alone is misleading across days).
            'label_for_charts': 'Exact match (date and hour)',
        },
        {
            'sql_condition': """
                ABS(
                    ((CAST(admit_date_r AS DATE) - CAST(admit_date_l AS DATE)) * 24)
                    + (CAST(admit_hr_r AS FLOAT) - CAST(admit_hr_l AS FLOAT))
                ) <= 48
            """,
            'label_for_charts': 'Within 48 hours (combined date and hour)',
        },
        {
            'sql_condition': 'TRUE',
            'label_for_charts': 'all other comparisons'
        }
    ]
)

In [55]:
# Choose comparison methods you wish to use (e.g. ExactMatch)
comparisons = [
        cl.ExactMatch('firstnamef2'), # add jaro winkler
        cl.ExactMatch('lastnamef2'),
        cl.ExactMatch('sex'),
        cl.ExactMatch('dob'),
        cl.ExactMatch('age'),
        cl.ExactMatch('county'),


        admit_date_comp,
        admit_hour_comp,
    ]
    #### Choose Comparison methods for each linkage variable -----------------------


In [56]:
#### Put together settings and create Linker object ----------------------------

# Step 12: Define full model settings
model_settings = SettingsCreator(
    unique_id_column_name = 'unique_id',
    link_type='link_only',
    blocking_rules_to_generate_predictions=blocking_rules,
    comparisons=comparisons,
    retain_intermediate_calculation_columns=True,
)

linker = Linker(
    [df_crash, df_hosp],
    model_settings,
    db_api = DuckDBAPI()
)

#### Parameter Estimation ------------------------------------------------------

# Step 13: Pre-training — estimate lambda (probability two random records match)
deterministic_rules = [
    block_on('firstnamef2', 'lastnamef2'),
    block_on('dob', 'age', 'sex'),
    block_on('lastnamef2', 'age'),
]

linker.training.estimate_probability_two_random_records_match(
    deterministic_rules, recall=0.8
)

# Step 14: Pre-training — estimate u probabilities using random sampling
# Estimate the prior using a direct estimation technique
# It's recommended to set max pairs high (e.g. 1e7 or more)
# Estimate U probabilities with random sampling
linker.training.estimate_u_using_random_sampling(max_pairs=1e7)


INFO:splink.internals.linker_components.training:Probability two random records match is estimated to be  0.000871.
This means that amongst all possible pairwise record comparisons, one in 1,147.78 are expected to match.  With 1,000,000 total possible comparisons, we expect a total of around 871.25 matching pairs
INFO:splink.internals.estimate_u:----- Estimating u probabilities using random sampling -----
INFO:splink.internals.estimate_u:
Estimated u probabilities using random sampling
INFO:splink.internals.settings:
Your model is not yet fully trained. Missing estimates for:
    - firstnamef2 (no m values are trained).
    - lastnamef2 (no m values are trained).
    - sex (no m values are trained).
    - dob (no m values are trained).
    - age (no m values are trained).
    - county (no m values are trained).
    - admit_date_comp (no m values are trained).
    - admit_hr_comp (no m values are trained).


In [57]:
# Step 15: EM Training — estimate m probabilities

# Estimate M probabilities with EM algorithm -- requires multiple sessions
# Each session blocks on different variables so all comparisons get trained
df_crash = df_crash.rename(columns={'splink_id': 'unique_id'})
df_hosp  = df_hosp.rename(columns={'splink_id': 'unique_id'})


In [58]:
#Training sessions


session_1 = linker.training.estimate_parameters_using_expectation_maximisation(

block_on('lastnamef2', 'age'), estimate_without_term_frequencies=True

)



INFO:splink.internals.em_training_session:
----- Starting EM training session -----

INFO:splink.internals.em_training_session:Estimating the m probabilities of the model by blocking on:
(l."lastnamef2" = r."lastnamef2") AND (l."age" = r."age")

Parameter estimates will be made for the following comparison(s):
    - firstnamef2
    - sex
    - dob
    - county
    - admit_date_comp
    - admit_hr_comp

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - lastnamef2
    - age
INFO:splink.internals.expectation_maximisation:
INFO:splink.internals.expectation_maximisation:Iteration 1: Largest change in params was -0.705 in the m_probability of admit_date_comp, level `Exact match on admit_date`
INFO:splink.internals.expectation_maximisation:Iteration 2: Largest change in params was 0.0301 in the m_probability of admit_date_comp, level `all other comparisons`
INFO:splink.internals.expectation_maximisation:Iteration 3: Largest ch

In [59]:
session_2 = linker.training.estimate_parameters_using_expectation_maximisation(

block_on('firstnamef2', 'county'), estimate_without_term_frequencies=True

)


INFO:splink.internals.em_training_session:
----- Starting EM training session -----

INFO:splink.internals.em_training_session:Estimating the m probabilities of the model by blocking on:
(l."firstnamef2" = r."firstnamef2") AND (l."county" = r."county")

Parameter estimates will be made for the following comparison(s):
    - lastnamef2
    - sex
    - dob
    - age
    - admit_date_comp
    - admit_hr_comp

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - firstnamef2
    - county
INFO:splink.internals.expectation_maximisation:
INFO:splink.internals.expectation_maximisation:Iteration 1: Largest change in params was -0.114 in probability_two_random_records_match
INFO:splink.internals.expectation_maximisation:Iteration 2: Largest change in params was -0.00263 in the m_probability of age, level `All other comparisons`
INFO:splink.internals.expectation_maximisation:Iteration 3: Largest change in params was 0.000528 in the m_

In [60]:
session_3 = linker.training.estimate_parameters_using_expectation_maximisation(

block_on('firstnamef2', 'lastnamef2'), estimate_without_term_frequencies=True

)


INFO:splink.internals.em_training_session:
----- Starting EM training session -----

INFO:splink.internals.em_training_session:Estimating the m probabilities of the model by blocking on:
(l."firstnamef2" = r."firstnamef2") AND (l."lastnamef2" = r."lastnamef2")

Parameter estimates will be made for the following comparison(s):
    - sex
    - dob
    - age
    - county
    - admit_date_comp
    - admit_hr_comp

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - firstnamef2
    - lastnamef2
INFO:splink.internals.expectation_maximisation:
INFO:splink.internals.expectation_maximisation:Iteration 1: Largest change in params was -0.136 in probability_two_random_records_match
INFO:splink.internals.expectation_maximisation:Iteration 2: Largest change in params was -0.00141 in the m_probability of admit_date_comp, level `all other comparisons`
INFO:splink.internals.expectation_maximisation:Iteration 3: Largest change in params wa

In [61]:
# Step 16: Visualize model parameters

linker.evaluation.unlinkables_chart()

alt.LayerChart(...)

In [62]:
linker.visualisations.match_weights_chart()


alt.VConcatChart(...)

In [63]:
linker.visualisations.m_u_parameters_chart()

alt.HConcatChart(...)

In [64]:
# Step 17: Predict — run linkage and get match probabilities (all blocked pairs)
df_predictions = linker.inference.predict()
df_predictions.as_pandas_dataframe(limit=5)

INFO:splink.internals.linker_components.inference:Blocking time: 0.19 seconds
INFO:splink.internals.linker_components.inference:Predict time: 0.34 seconds


,match_weight,match_probability,source_dataset_l,source_dataset_r,unique_id_l,unique_id_r,firstnamef2_l,firstnamef2_r,gamma_firstnamef2,bf_firstnamef2,...,bf_county,admit_date_l,admit_date_r,gamma_admit_date_comp,bf_admit_date_comp,admit_hr_l,admit_hr_r,gamma_admit_hr_comp,bf_admit_hr_comp,match_key
0,-17.368391,0.000006,__splink__input_table_0,__splink__input_table_1,crash_742,hosp_1,Kenneth,Kenneth,1,55.464953,...,9.304043,2024-02-17,2024-02-24,0,0.263541,10,0,0,0.539199,3
1,-17.368391,0.000006,__splink__input_table_0,__splink__input_table_1,crash_985,hosp_2,Edward,Edward,1,55.464953,...,9.304043,2024-09-05,2024-07-14,0,0.263541,13,5,0,0.539199,3
2,-18.102927,0.000004,__splink__input_table_0,__splink__input_table_1,crash_516,hosp_3,Sandra,Sandra,1,55.464953,...,0.237239,2024-06-07,2024-06-16,0,0.263541,17,23,0,0.539199,4
3,12.489858,0.999826,__splink__input_table_0,__splink__input_table_1,crash_4,hosp_4,Timothy,Timotiy,0,0.113613,...,9.304043,2024-11-01,2024-11-02,1,45.674152,2,14,0,0.539199,1
4,-17.368391,0.000006,__splink__input_table_0,__splink__input_table_1,crash_134,hosp_5,Steven,Steven,1,55.464953,...,9.304043,2024-10-29,2024-09-29,0,0.263541,2,21,0,0.539199,3


In [66]:
df_predictions_pd = df_predictions.as_pandas_dataframe()

In [67]:
df_filtered = df_predictions_pd[df_predictions_pd['match_probability'] >= 0.8]

cols_keep = [
    'unique_id_l', 'unique_id_r',
    'match_probability', 'match_weight', 'match_key',
    'firstnamef2_l', 'lastnamef2_l', 'firstnamef2_r', 'lastnamef2_r',
    'dob_l', 'sex_l', 'age_l',
    'dob_r', 'sex_r', 'age_r',
    'county_l', 'county_r'
    'role_l', 'role_r',
    'admit_date_l', 'admit_date_r',
    'admit_hr_l', 'admit_hr_r',
]

df_filtered = df_filtered[[c for c in cols_keep if c in df_filtered.columns]]
df_filtered.to_csv('synthetic_linkage_results.csv', index=False)
print(f"Matched records saved: {len(df_filtered)} rows")

Matched records saved: 310 rows
